# colony_board

> Phase 18, Stage 4 (`specs/multiagent/phase18/stage4.md`): the board options that let guards scale. `ColonyBoardEnv` is the clue board with guard copies (`guardDeal`: one copy of every clue goes to a guard), squads (`squads`: subtasks and their clues stay within squads of S agents) and a record of wrong answers (`failFlag`); `failures` counts, for each posted value, the wrong answers that agreed with it from a submitter who did not hold that clue. `squadView` is one agent's view of the board (its squad's subtasks only), `InSquad` a numpy member acting on that view, and `stage4Members` applies both (and flag-aware known clues) to a run's numpy members. Built only when a Stage 4 key is set; every earlier board is a plain `ClueBoardEnv`, unchanged.

In [ ]:
#| default_exp colony_board

In [ ]:
#| hide
from nbdev.showdoc import *

## The board

In [ ]:
#| export
from functools import lru_cache

from explRL.clue_agents import RuleAgent
from explRL.clueboard import ClueBoardEnv

STAGE4_KEYS = ('guardDeal', 'failFlag', 'squads')


def squadOf(i, N, S):
    "Slot i's squad: i mod (N // S)."
    return i % (N // S)


class ColonyBoardEnv(ClueBoardEnv):
    """The clue board with Stage 4's options. guardDeal: per clue one copy to a guard slot (`guardSlots`), one to another
    slot, each drawn as the base deal draws (from the subtask's free slots, `rngDeal`); squads (S): subtask k belongs to a
    squad (the 2N + 2 first subtasks round-robin, a replacement to the squad of the subtask it replaces) and its clues are
    dealt within that squad; failFlag: wrong answers (t, i, k, x) are kept in `failed`. All off: the base deal exactly."""
    def __init__(self, seed, nAgents, guardSlots=range(0), guardDeal=False, squads=None, failFlag=False, **kw):
        self.guardSet, self.guardDeal, self.failFlag = frozenset(guardSlots), bool(guardDeal), bool(failFlag)
        self.S = int(squads) if squads else None
        if self.S and nAgents % self.S:
            raise ValueError(f'squads: N = {nAgents} is not a multiple of S = {self.S}')
        if self.guardDeal and (len(self.guardSet) < 2 or kw.get('r', 1) != 2):
            raise ValueError('guardDeal needs at least two guard slots and r = 2')
        self.nSquads = nAgents // self.S if self.S else 1
        super().__init__(seed, nAgents, **kw)

    def reset(self, seed, nAgents):
        M = 2 * int(nAgents) + 2
        self.failed, self.squadOfTask, self._squad = [], {}, 0
        self._target = [M // self.nSquads + (s < M % self.nSquads) for s in range(self.nSquads)]
        super().reset(seed, nAgents)

    def _new(self, t):
        cnt = [0] * self.nSquads
        for k in self.open:
            cnt[self.squadOfTask[k]] += 1
        self._squad = max(range(self.nSquads), key=lambda s: (self._target[s] - cnt[s], -s))
        self.squadOfTask[self.serial] = self._squad
        super()._new(t)

    def _pick(self, cand, held):
        safe = [a for a in cand if held[a] < self.m - 1] or cand
        lo = min(held[a] for a in safe)
        pool = [a for a in safe if held[a] == lo]
        return int(pool[int(self.rngDeal.integers(len(pool)))])

    def _deal(self):
        if not (self.guardDeal or self.S):
            return super()._deal()
        slots = [a for a in range(self.nAgents) if not self.S or a % self.nSquads == self._squad]
        held, out = {a: 0 for a in slots}, []
        for _ in range(self.m):
            hs = set()
            groups = ([[a for a in slots if a in self.guardSet], [a for a in slots if a not in self.guardSet]]
                      if self.guardDeal else [slots] * min(self.r, len(slots)))
            for g in groups:
                a = self._pick([a for a in g if a not in hs], held)
                hs.add(a)
                held[a] += 1
            out.append(frozenset(hs))
        return out

    def apply(self, i, action, t):
        if self.S and action[0] in ('open', 'share'):
            k = action[1] if action[0] == 'open' else action[1][0]
            if k in self.squadOfTask and self.squadOfTask[k] != i % self.nSquads:
                raise ValueError(f"agent {i} acted on subtask {k} of another squad")
        n = len(self.submits)
        out = super().apply(i, action, t)
        if self.failFlag and len(self.submits) > n and self.submits[-1][4] == 'wrong':
            self.failed.append((float(t), i, self.submits[-1][2], int(action[1])))
        return out


def failures(board):
    """{(k, j, r): n}: n wrong answers to k agreed with the posted value r of clue j (x mod p_j == r) from a submitter
    who does not hold clue j; posted values are the board's shares, whenever posted. {} without failFlag."""
    if not getattr(board, 'failFlag', False) or not board.failed:
        return {}
    posted, out = {}, {}
    for s in board.shareLog:
        posted.setdefault((s[2], s[3]), set()).add(s[4])
    for _, i, k, x in board.failed:
        s = board.tasks[k]
        for j, hs in enumerate(s.holders):
            if i not in hs:
                for r in posted.get((k, j), ()):
                    if x % s.task.primes[j] == r:
                        out[(k, j, r)] = out.get((k, j, r), 0) + 1
    return out

## Squad views and members

In [ ]:
#| export
class SquadView:
    "Agent i's view of a squads board: `open` and `observe(i)['open']` hold its squad's subtasks only; all else is the board."
    def __init__(self, board, i):
        self.__dict__.update(_b=board, _s=i % board.nSquads)

    @property
    def open(self):
        return [k for k in self._b.open if self._b.squadOfTask[k] == self._s]

    def observe(self, i):
        o, keep = self._b.observe(i), set(self.open)
        return dict(o, open=[x for x in o['open'] if x[0] in keep])

    def __getattr__(self, name):
        return getattr(self._b, name)


def squadView(board, i):
    "The squad view of `board` for agent i, or the board itself when it has no squads."
    return SquadView(board, i) if getattr(board, 'S', None) else board


class InSquad:
    "A numpy member that sees its squad's view of the board (its id, extras and everything else are the member's)."
    def __init__(self, member):
        self.__dict__['m'] = member

    @property
    def id(self):
        return self.m.id

    @id.setter
    def id(self, v):
        self.m.id = v

    def __getattr__(self, name):
        return getattr(self.m, name)

    def event(self, world, t):
        return self.m.event(squadView(world, self.m.id), t)


def _flagKnown(self, b):
    "ColonyAgent.knownFor, the leading confirmed value per clue taken among the values with no failure."
    from explRL.clue_colony import tally
    fl, tal, out = failures(b), tally(self.channel, b), {k: {} for k in b.open}
    for k in out:
        for j, vs in tal.get(k, {}).items():
            ok = [v for v in vs if not fl.get((k, j, v[0]))]
            if ok and len(ok[0][1]) >= self.q:
                out[k][j] = ok[0][0]
        out[k].update(self.heldFor(b, k))
    return out


@lru_cache(None)
def _flagged(cls):
    return type('Flag' + cls.__name__, (cls,), dict(knownFor=_flagKnown))


def stage4Members(world, members):
    """A run's members with Stage 4's options: with failFlag, colony rule agents read failures (their class gains the
    flag-aware knownFor); with squads, every numpy member (scripted liars included) acts through InSquad. Claude members are
    left alone (TextClueEnv gives them the squad view). No Stage 4 key: the members unchanged."""
    from explRL.colony_agents import ColonyAgent
    out = []
    for m in members:
        if world.get('failFlag') and isinstance(m, ColonyAgent) and not type(m).__name__.startswith('Flag'):
            m.__class__ = _flagged(type(m))
        out.append(InSquad(m) if world.get('squads') and isinstance(m, RuleAgent) else m)
    return out

## Tests

In [ ]:
import numpy as np

from explRL.clueboard import ClueBoardEnv

kw = dict(m=2, r=2, R=30, value=(1, 2, 3), life=30, penalty=0.5)
# all options off: the same subtasks and deal as the base board (the deal stream untouched)
a, b = ClueBoardEnv(7, 16, **kw), ColonyBoardEnv(7, 16, **kw)
assert all(a.tasks[k].holders == b.tasks[k].holders and a.tasks[k].task == b.tasks[k].task for k in a.tasks)

# guardDeal: per clue one guard and one non-guard holder, never both clues to one agent
G = range(4, 8)
for seed in range(20):
    g = ColonyBoardEnv(seed, 16, guardSlots=G, guardDeal=True, **kw)
    for _ in range(40):                                   # replacements deal too
        g.advance(g.tWorld + 1)
    for s in g.tasks.values():
        assert all(len(hs & set(G)) == 1 and len(hs) == 2 for hs in s.holders)
        assert not (s.holders[0] & s.holders[1])
# subtasks and values unchanged by the deal
assert all(g.tasks[k].value == ColonyBoardEnv(19, 16, **kw).tasks[k].value for k in range(34))

In [ ]:
# squads: squad make-up (2 liars, 2 guards, 4 others per squad of 8), deals within the squad, fixed counts per squad
for N in (16, 64):
    nS, nL = N // 8, N // 4
    for s in range(nS):
        members = [i for i in range(N) if squadOf(i, N, 8) == s]
        assert len(members) == 8 and sum(i < nL for i in members) == 2 and sum(nL <= i < 2 * nL for i in members) == 2
    q = ColonyBoardEnv(3, N, guardSlots=range(nL, 2 * nL), guardDeal=True, squads=8, **kw)
    for _ in range(35):
        q.advance(q.tWorld + 1)
        cnt = np.bincount([q.squadOfTask[k] for k in q.open], minlength=nS)
        assert sorted(cnt.tolist()) == sorted(q._target), (cnt, q._target)
    for k, s in q.tasks.items():
        assert all(h % nS == q.squadOfTask[k] for hs in s.holders for h in hs)
        assert all(len(hs & set(range(nL, 2 * nL))) == 1 for hs in s.holders)

# the squad view hides other squads' subtasks; acting on one raises
v = squadView(q, 5)
assert set(v.open) == {k for k in q.open if q.squadOfTask[k] == 5 % (64 // 8)} and v.R == q.R
assert all(x[0] in v.open for x in v.observe(5)['open']) and squadView(a, 5) is a
other = next(k for k in q.open if q.squadOfTask[k] != 5 % 8)
try:
    q.apply(5, ('open', other), q.tWorld)
    raise AssertionError('should raise')
except ValueError:
    pass

In [ ]:
# failures: a wrong answer flags only values of the clue the submitter does not hold
f = ColonyBoardEnv(11, 8, failFlag=True, **kw)
k = f.open[0]
s = f.tasks[k]
p0, p1 = s.task.primes
i = next(iter(s.holders[0]))                              # holds clue 0 (true), not clue 1
other = next(a for a in range(8) if a not in s.holders[0] | s.holders[1])
true1 = s.task.clue(1)[1]
lie1 = next(r for r in range(p1) if r != true1 and any(x % p0 == s.task.clue(0)[1] and x % p1 == r for x in range(30)))
f.apply(other, ('share', (k, 1, lie1)), 0.0)             # a lie on clue 1
f.apply(other, ('share', (k, 1, true1)), 0.0)            # and the true value
x = next(x for x in range(30) if x % p0 == s.task.clue(0)[1] and x % p1 == lie1)   # consistent with own clue 0, built on the lie
f.apply(i, ('open', k), 1.0)
f.apply(i, ('submit', x), 2.0)
assert f.failed == [(2.0, i, k, x)]
assert failures(f) == {(k, 1, lie1): 1}                   # the true value of clue 1 and own clue 0 are never flagged
assert failures(ColonyBoardEnv(11, 8, **kw)) == {}

In [ ]:
# InSquad members never act outside their squad; stage4Members leaves members alone without Stage 4 keys
from explRL.clue_agents import ClueBoard, ClueLiar
from explRL.colony_agents import ColonyAgent, GuardAgent
from explRL.llm_agent import runTurns
from explRL.multi import spawnRngs

N, nL = 16, 4
w = dict(colony=True, guards=4, squads=8, failFlag=True, guardDeal=True)
bd, ch = ColonyBoardEnv(5, N, guardSlots=range(4, 8), guardDeal=True, squads=8, failFlag=True, **kw), ClueBoard()
rngs = spawnRngs(9, N)
mem = [ClueLiar(ch, rngs[i]) if i < nL else (GuardAgent if i < 8 else ColonyAgent)(ch, rngs[i]) for i in range(N)]
wrapped = stage4Members(w, mem)
assert all(isinstance(m, InSquad) for m in wrapped) and type(mem[10]).__name__ == 'FlagColonyAgent'
runTurns(wrapped, bd, 30)                                 # would raise on a cross-squad open or share
assert all(bd.squadOfTask[s[2]] == s[1] % 2 for s in bd.shareLog)
plain = [ClueLiar(ch, rngs[0])]
assert stage4Members({}, plain) == plain and type(plain[0]) is ClueLiar